# 11 — Qualitative Interview Analysis

Analyses the thematic analysis results from 14 practitioner interviews
(November–December 2024). Reproduces the qualitative findings from Section 5
and the practitioner evidence cited throughout the paper.

**14 participants** across Financial Services, Healthcare, Legal Tech,
E-Commerce, Software Dev Tooling, and Manufacturing.
**Inter-rater reliability**: Cohen's κ = 0.81 (substantial agreement).


In [ ]:
import sys, os, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

metadata = pd.read_csv('../interviews/interview_metadata.csv')
excerpts = pd.read_csv('../interviews/thematic_analysis/coded_excerpts.csv')
themes   = json.load(open('../interviews/thematic_analysis/theme_summary.json'))
iaa      = json.load(open('../interviews/thematic_analysis/iaa_analysis.json'))

print(f"Participants: {len(metadata)}")
print(f"Coded excerpts: {len(excerpts)}")
print(f"Themes: {themes['total_themes']}")
print(f"IAA Cohen κ: {iaa['overall_kappa']}")
print()
print(metadata[['interview_id','sector','participant_role','duration_minutes']].to_string(index=False))

Participants: 14
Coded excerpts: 237
Themes: 8
IAA Cohen κ: 0.81

 interview_id              sector                  participant_role  duration_minutes
     INT-01  Financial Services           Principal ML Engineer                62
     INT-02          Healthcare              AI Product Manager                54
     INT-03          Legal Tech        Head of LLM Engineering                71
     INT-04         E-Commerce          Senior Data Scientist                49
     INT-05  Software Dev Tooling         ML Platform Engineer                58
     INT-06       Manufacturing                 Applied AI Lead                63
     INT-07  Financial Services           LLM Safety Engineer                55
     INT-08          Healthcare     Clinical Informatics Lead                67
     INT-09          Legal Tech             VP of Engineering                48
     INT-10         E-Commerce        ML Engineering Manager                61
     INT-11  Software Dev Tooling     Sta

## 1. Theme frequency and sector distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Theme excerpt counts
theme_ids   = [f"TH{i}" for i in range(1,9)]
theme_names = ['Existing QA\ninadequacy','Hallucination\nconcern','Transparency\ngap',
               'Integration\nreliability','Threshold\ncalibration','Prompt\nbrittleness',
               'Regulatory\ndriver','Tooling\nfragmentation']
theme_counts = excerpts.groupby('theme_id').size().reindex(theme_ids).fillna(0)
colors_th = ['#e74c3c' if i in [1,2] else '#3498db' if i in [3,6] else '#2ecc71'
             for i in range(8)]
bars = axes[0].bar(theme_ids, theme_counts, color=colors_th, edgecolor='white')
axes[0].set_xticklabels([f"TH{i}" for i in range(1,9)], fontsize=10)
axes[0].set_ylabel('Coded excerpts'); axes[0].set_title('Excerpt Count per Theme')
for bar, n in zip(bars, theme_counts):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 str(int(n)), ha='center', va='bottom', fontsize=9)
axes[0].grid(axis='y', alpha=0.3)

# Sector distribution
sector_counts = metadata['sector'].value_counts()
wedges, texts, autotexts = axes[1].pie(
    sector_counts.values, labels=sector_counts.index, autopct='%1.0f%%',
    pctdistance=0.75, startangle=140,
    colors=['#4C72B0','#DD8452','#55A868','#C44E52','#9B59B6','#1ABC9C'])
axes[1].set_title('Participant Sector Distribution  (n=14)')

plt.suptitle('Practitioner Interview Analysis  (14 interviews, Nov–Dec 2024)', fontsize=13)
plt.tight_layout()
plt.show()

print(f"\nMost common theme: TH2 (Hallucination) — {int(theme_counts['TH2'])} excerpts")
print(f"Least common theme: TH8 (Tooling fragmentation) — {int(theme_counts['TH8'])} excerpts")


Most common theme: TH2 (Hallucination) — 52 excerpts
Least common theme: TH8 (Tooling fragmentation) — 8 excerpts


## 2. TH3 Transparency — the most actionable finding

In [ ]:
th3 = json.load(open('../interviews/thematic_analysis/TH3_theme_report.json'))

print("TH3: Transparency as Underserved Quality Gap")
print("=" * 55)
print(f"Excerpts: {th3['n_excerpts']}")
print(f"Significance: {th3['significance_for_qalis']}")
print()
print("Key evidence:")
for e in th3['key_evidence']:
    print(f"  • {e}")
print()
print("Paper connection: TI was the lowest-scoring QALIS dimension")
print("  Mean TI = 7.05, σ = 1.35 (largest variance across systems)")
print("  11/14 participants cited transparency gap unprompted")

# TI scores per system
ti_scores = {'S1':5.6,'S2':6.2,'S3':7.5,'S4':8.9}
fig, ax = plt.subplots(figsize=(7, 3.5))
colors_ti = ['#e74c3c' if v < 7.0 else '#f39c12' if v < 8.0 else '#2ecc71'
             for v in ti_scores.values()]
ax.bar(ti_scores.keys(), ti_scores.values(), color=colors_ti, edgecolor='white')
ax.axhline(7.05, color='grey', linestyle='--', linewidth=1.2, label='Study mean (7.05)')
ax.axhline(7.5, color='black', linestyle=':', linewidth=0.8, label='Adequate threshold (7.5)')
ax.set_ylim(0, 10); ax.set_ylabel('TI Score (0–10)')
ax.set_title('TI Transparency Scores — S1 to S4\n(lowest-scoring dimension, σ=1.35)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

TH3: Transparency as Underserved Quality Gap
Excerpts: 48
Significance: Second most common theme (n=48). Paper finding: TI lowest-scoring QALIS dimension (mean 7.05). 11/14 participants cited transparency gap unprompted. Directly motivates TI dimension.

Key evidence:
  • Audit trails exist but are unstructured and unqueryable (INT-01, INT-07, INT-13)
  • User trust erosion when outputs cannot be explained — users stop relying on the system (INT-04, INT-10)
  • Regulatory explanation gap: EU AI Act, HIPAA, and financial regulation all require explainability (INT-07, INT-08, INT-13, INT-14)

Paper connection: TI was the lowest-scoring QALIS dimension
  Mean TI = 7.05, σ = 1.35 (largest variance across systems)
  11/14 participants cited transparency gap unprompted


## 3. Interview duration and IAA summary

In [ ]:
print(f"Interview durations:")
print(f"  Min: {metadata['duration_minutes'].min()} min")
print(f"  Max: {metadata['duration_minutes'].max()} min")
print(f"  Mean: {metadata['duration_minutes'].mean():.0f} min")
print()
print("Per-theme IAA (Cohen's κ):")
for th_id, kappa in iaa['per_theme_kappa'].items():
    status = '✓' if kappa >= 0.70 else '!'
    print(f"  {status} {th_id}: κ={kappa:.2f}")
print(f"\nOverall κ={iaa['overall_kappa']:.2f} — {iaa['interpretation']}")

Interview durations:
  Min: 48 min
  Max: 71 min
  Mean: 58 min

Per-theme IAA (Cohen's κ):
  ✓ TH1: κ=0.84
  ✓ TH2: κ=0.79
  ✓ TH3: κ=0.83
  ✓ TH4: κ=0.82
  ✓ TH5: κ=0.78
  ✓ TH6: κ=0.80
  ✓ TH7: κ=0.85
  ✓ TH8: κ=0.77

Overall κ=0.81 — Substantial agreement (Landis & Koch, 1977 scale)
